In [1]:
import random
import gymnasium as gym
import numpy as np
import collections
from tqdm import tqdm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
#import drive.MyDrive.rl_utils
import rl_utils

In [2]:
class ReplayBuffer:
    ''' 经验回放池 '''
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)  # 队列,先进先出

    def add(self, state, action, reward, next_state, done):  # 将数据加入buffer
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):  # 从buffer中采样数据,数量为batch_size
        transitions = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = zip(*transitions)
        return np.array(state), action, reward, np.array(next_state), done

    def size(self):  # 目前buffer中数据的数量
        return len(self.buffer)

In [3]:
class Qnet(torch.nn.Module):
    def __init__(self, state_dim, hidden_dim, action_dim):
        super().__init__()
        self.fc1 = torch.nn.Linear(state_dim, hidden_dim)
        self.fc2 = torch.nn.Linear(hidden_dim, action_dim)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [4]:
class DQN:
    
    def __init__(self, state_dim, hidden_dim, action_dim, learning_rate, gamma
                 , epsilon, target_update, device):
        
        self.action_dim = action_dim
        self.q_net = Qnet(state_dim, hidden_dim, action_dim).to(device)
        
        # 反正是垃圾目标，参数一不一样无所谓
        self.target_q_net = Qnet(state_dim, hidden_dim, action_dim).to(device)
        
        self.optimizer = torch.optim.Adam(self.q_net.parameters(),
                                          lr = learning_rate)
        
        self.gamma = gamma  # 折扣因子
        self.epsilon = epsilon  # epsilon-贪婪策略
        self.target_update = target_update  # 目标网络更新频率
        self.count = 0  # 计数器,记录更新次数
        self.device = device
        
    def take_action(self, state):    
        if np.random.random() < self.epsilon:
            action = np.random.randint(self.action_dim)
        else:
            state = torch.tensor([state], dtype = torch.float).to(self.device)
            with torch.no_grad():
                action = self.q_net(state).argmax(dim=1).item()
        
        return action
    
    def update(self, transition_dict):
        states = torch.tensor(transition_dict['states'],
                              dtype=torch.float).to(self.device)
        actions = torch.tensor(transition_dict['actions'], dtype=torch.long).view(-1, 1).to(
            self.device)
        rewards = torch.tensor(transition_dict['rewards'],
                               dtype=torch.float).view(-1, 1).to(self.device)
        next_states = torch.tensor(transition_dict['next_states'],
                                   dtype=torch.float).to(self.device)
        dones = torch.tensor(transition_dict['dones'],
                             dtype=torch.float).view(-1, 1).to(self.device)

        # tensor.gather(dim, index) 按dim方向 index位置取
        # 即取出 q_k, a_kß
        q_values = self.q_net(states).gather(1, actions)
        
        #下个状态最大q值
        # tensor.max() return {values, indices}
        with torch.no_grad():
            max_next_q_values = self.target_q_net(next_states).max(dim=1, keepdim=True).values
            
            # 在终止状态未来无价值
            q_targets = rewards + self.gamma * max_next_q_values * (1 - dones)
        
        # F.mse_loss 输出标量
        dqn_loss = F.mse_loss(q_values, q_targets)
        
        self.optimizer.zero_grad()
        dqn_loss.backward()
        self.optimizer.step()
        
        
        self.count += 1
        
        if self.count % self.target_update == 0:
            # 更新目标网络
            self.target_q_net.load_state_dict(
                self.q_net.state_dict()
            )
        
        

In [5]:
def train_dqn(agent, env, replay_buffer,
              num_episodes,
              minimal_size,
              batch_size):

    return_list = []
    best_return = 0

    for i in range(10):
        with tqdm(
            total=num_episodes // 10,
            desc=f"Iteration {i}"
        ) as pbar:

            for i_episode in range(num_episodes // 10):

                episode_return = 0

                state, info = env.reset()

                done = False

                while not done:

                    action = agent.take_action(state)

                    next_state, reward, terminated, truncated, info = \
                        env.step(action)

                    done = terminated or truncated


                    replay_buffer.add(
                        state,
                        action,
                        reward,
                        next_state,
                        done
                    )


                    state = next_state
                    episode_return += reward


                    # replay buffer足够大才训练
                    if replay_buffer.size() > minimal_size:

                        b_s, b_a, b_r, b_ns, b_d = \
                            replay_buffer.sample(batch_size)


                        transition_dict = {
                            "states": b_s,
                            "actions": b_a,
                            "next_states": b_ns,
                            "rewards": b_r,
                            "dones": b_d
                        }

                        agent.update(transition_dict)


                return_list.append(episode_return)

                if episode_return > best_return:
                        best_return = episode_return
                        torch.save(
                            agent.q_net.state_dict(),
                            "best.pth"
                        )
                
                if (i_episode + 1) % 10 == 0:

                    avg_return = np.mean(return_list[-10:])

                    pbar.set_postfix(
                        {
                            "episode":
                            i * (num_episodes//10)
                            + i_episode + 1,

                            "return":
                            f"{avg_return:.2f}"
                        }
                    )


                pbar.update(1)

    
    
    return return_list

In [11]:
def evaluate_dqn(agent, env, episodes=5):

    # 关闭探索
    old_epsilon = agent.epsilon
    agent.epsilon = 0


    for episode in range(episodes):

        state, info = env.reset()

        done = False
        total_reward = 0


        while not done:

            action = agent.take_action(state)


            next_state, reward, terminated, truncated, info = \
                env.step(action)


            done = terminated or truncated

            state = next_state

            total_reward += reward


        print(
            f"Episode {episode+1}: reward={total_reward}"
        )


    # 恢复
    agent.epsilon = old_epsilon
    env.close()
    

In [7]:
train_env = gym.make(
    "CartPole-v1"
)

In [8]:
lr = 2e-3
hidden_dim = 128
gamma = 0.98
epsilon = 0.1
target_update = 10

buffer_size = 10000
minimal_size = 500
batch_size = 64


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


state_dim = train_env.observation_space.shape[0]
action_dim = train_env.action_space.n


agent = DQN(
    state_dim,
    hidden_dim,
    action_dim,
    lr,
    gamma,
    epsilon,
    target_update,
    device
)


replay_buffer = ReplayBuffer(buffer_size)

In [9]:
return_list = train_dqn(
    agent,
    train_env,
    replay_buffer,
    num_episodes=500,
    minimal_size=500,
    batch_size=64
)

Iteration 0:   0%|          | 0/50 [00:00<?, ?it/s]/var/folders/zs/b42fwrrn0yx49kwcgrgmq4c00000gn/T/ipykernel_9263/528853318.py:25: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:255.)
  state = torch.tensor([state], dtype = torch.float).to(self.device)
Iteration 9: 100%|██████████| 50/50 [00:04<00:00, 11.03it/s, episode=500, return=429.40]


In [13]:
test_env = gym.make(
    "CartPole-v1",
    render_mode="human"
)

agent.q_net.load_state_dict(
    torch.load(
        "best.pth",
        weights_only=True
    )
)

evaluate_dqn(
    agent,
    test_env,
    episodes=5
)

Episode 1: reward=500.0
Episode 2: reward=500.0
Episode 3: reward=402.0
Episode 4: reward=365.0
Episode 5: reward=500.0
